In [1]:
%reset -f


In [2]:
import numpy as np
import pandas as pd
import torch
from pathlib import Path

import sys
sys.path.append("../../tools")
import prepare as prep
import functions as func

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [3]:
import importlib
#importlib.reload(acht)


In [4]:
load_dir1 = Path(f"../../saved_models/sta_entropy_15fold/sp3to20_MD_transpt3_r9")
load_dir2 = Path(f"../../saved_models/diversity_15fold/sp3to20_MD_transpt3_r1")
load_dir3 = Path(f"../../saved_models/num_stable_states_15fold/sp3to20_NS_transpt3_r7")
load_dir4 = Path(f"../../saved_models/fluctuation_15fold_v2/sp3to20_Fl_transpt3_r1")
load_dir5 = Path(f"../../saved_models/swi_species_15fold/sp3to20_SS_transpt3_r13")
emodel = func.load_saved_model_MD(load_dir1, device)
dmodel = func.load_saved_model_MD(load_dir2, device)
nmodel = func.load_saved_model_NS(load_dir3, device)
fmodel = func.load_saved_model_MD(load_dir4, device)
smodel = func.load_saved_model_SS(load_dir5, device)
epredictor = func.Predictor(emodel)
dpredictor = func.Predictor(dmodel)
npredictor = func.Predictor(nmodel)
fpredictor = func.Predictor(fmodel)
spredictor = func.Predictor(smodel)


c:\Users\HW\anaconda3\Lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.5.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


**The pos_rate, neg_center and density in gamma5 is more comprehensive**

In [ ]:
# Fix mu, D, randomize richness and gamma under different density and positive proportion conditions.
save_path = "gamma5.csv"

datasize = 8000
max_species = 20
all_data = []

for NumSpecies in range(3, max_species + 1):

    featgam = NumSpecies**2 - NumSpecies
    total_dim = NumSpecies + featgam + NumSpecies
    density = np.random.rand(datasize)
    pos_inall = np.random.rand(datasize)
    gamma_neg_cen = 0.1 + 6.9*np.random.rand(datasize)
    gamma_pos_cen = 0.1 + 6.9*np.random.rand(datasize)
    gamma_mean = np.zeros(datasize)
    n_edge = NumSpecies * (NumSpecies - 1) # Upper boundary of the number of interaction.
    Xnum = np.zeros((datasize, total_dim))

    den_arr = np.zeros(datasize)
    pos_arr = np.zeros(datasize)

    Xnum[:, 0:NumSpecies] = 0.5 + 0.1*np.random.randn(datasize, NumSpecies)

    for i in range(datasize):

        n_nonzero = int(round(density[i] * n_edge)) # None zero interaction number.
        n_pos = int(round(pos_inall[i] * n_nonzero)) # Positive interaction number.
        n_neg = n_nonzero - n_pos

        idx_nonzero = np.random.choice(n_edge, n_nonzero, replace=False) # None zero interaction index(from 0).
        idx_pos = np.random.choice(idx_nonzero, n_pos, replace=False)

        mask_pos = np.zeros(n_edge, dtype=bool)
        mask_pos[idx_pos] = True

        mask_neg = np.zeros(n_edge, dtype=bool)
        mask_neg[idx_nonzero] = True
        mask_neg[idx_pos] = False

        m_neg = 3
        m_pos = 3
        gamma_sigma = 0.2

        # Randomly generate neg-interaction coefficient according to negative number.
        neg_value = -m_neg * ((1 - gamma_sigma) + 2 * gamma_sigma * np.random.rand(n_neg))
        pos_value = m_pos * ((1 - gamma_sigma) + 2 * gamma_sigma * np.random.rand(n_pos))

        # Scaled individually to the target gamma mean.
        if n_neg > 0:
            neg_value *= gamma_neg_cen[i] / np.mean(np.abs(neg_value))
        if n_pos > 0:
            pos_value *= gamma_pos_cen[i] / np.mean(pos_value)
        if n_nonzero > 0:
            gamma_mean[i] = (np.sum(np.abs(neg_value))+np.sum(pos_value)) / n_nonzero # Only calculate non zero interaction.

        gammas = np.zeros(n_edge)
        gammas[mask_neg] = neg_value
        gammas[mask_pos] = pos_value

        Xnum[i, NumSpecies:NumSpecies + featgam] = gammas

        if n_nonzero > 0:
            den_arr[i] = n_nonzero / n_edge
            pos_arr[i] = n_pos / n_nonzero
        else:
            den_arr[i] = 0
            pos_arr[i] = 0

    Xnum[:, (NumSpecies + featgam):(NumSpecies + featgam + NumSpecies)] = \
        1e-06 / NumSpecies * (0.5 + 0.1 * np.random.randn(datasize, NumSpecies))
    
    # ===== padding =====
    Xnum_pad = prep.pad_Xnum_to_maxspecies(Xnum, NumSpecies, max_species)
    #Xnum_pad = prep.pad_Xnum_to_maxspecies(Xnum, NumSpecies, 20)

    richness_col = np.full(datasize, NumSpecies)

    gamma_neg_arr = gamma_neg_cen
    gamma_pos_arr = gamma_pos_cen
    gamma_mean_arr = gamma_mean

    relative_entropy = epredictor.predict_array_MD(Xnum_pad, device)
    entropy = relative_entropy*(NumSpecies-1)+1
    relative_diversity = dpredictor.predict_array_MD(Xnum_pad, device)
    diversity = relative_diversity*(NumSpecies-1)+1
    num_stable_states = npredictor.predict_array_NS(Xnum_pad, device)
    fluctuation = fpredictor.predict_array_Fl(Xnum_pad, device)
    swi_species = spredictor.predict_array_SS(Xnum_pad, device)

    extra = np.column_stack([richness_col, den_arr, pos_arr, gamma_neg_arr, gamma_pos_arr, gamma_mean_arr,
                                relative_entropy, entropy, relative_diversity, diversity, num_stable_states,
                                fluctuation, swi_species])

    X_full = np.concatenate([Xnum_pad, extra], axis=1)
    all_data.append(X_full)

    print(f"Processed NumSpecies = {NumSpecies}")


Processed NumSpecies = 3
Processed NumSpecies = 4
Processed NumSpecies = 5
Processed NumSpecies = 6
Processed NumSpecies = 7
Processed NumSpecies = 8
Processed NumSpecies = 9
Processed NumSpecies = 10
Processed NumSpecies = 11
Processed NumSpecies = 12
Processed NumSpecies = 13
Processed NumSpecies = 14
Processed NumSpecies = 15
Processed NumSpecies = 16
Processed NumSpecies = 17
Processed NumSpecies = 18
Processed NumSpecies = 19
Processed NumSpecies = 20


**gamma5_fluc has more fluctuation samples.**

In [5]:
# Fix mu, D, randomize richness and gamma under different density and positive proportion conditions.
save_path = "gamma5_fluc.csv"

datasize = 8000
max_species = 20
all_data = []

for NumSpecies in range(3, max_species + 1):

    featgam = NumSpecies**2 - NumSpecies
    total_dim = NumSpecies + featgam + NumSpecies
    density = 0.5 + 0.5*np.random.rand(datasize)
    pos_inall = 0.1 + 0.2*np.random.rand(datasize)
    gamma_neg_cen = 2 + 5*np.random.rand(datasize)
    gamma_pos_cen = 0.1 + 6.9*np.random.rand(datasize)
    gamma_mean = np.zeros(datasize)
    n_edge = NumSpecies * (NumSpecies - 1) # Upper boundary of the number of interaction.
    Xnum = np.zeros((datasize, total_dim))

    den_arr = np.zeros(datasize)
    pos_arr = np.zeros(datasize)

    Xnum[:, 0:NumSpecies] = 0.5 + 0.1*np.random.randn(datasize, NumSpecies)

    for i in range(datasize):

        n_nonzero = int(round(density[i] * n_edge)) # None zero interaction number.
        n_pos = int(round(pos_inall[i] * n_nonzero)) # Positive interaction number.
        n_neg = n_nonzero - n_pos

        idx_nonzero = np.random.choice(n_edge, n_nonzero, replace=False) # None zero interaction index(from 0).
        idx_pos = np.random.choice(idx_nonzero, n_pos, replace=False)

        mask_pos = np.zeros(n_edge, dtype=bool)
        mask_pos[idx_pos] = True

        mask_neg = np.zeros(n_edge, dtype=bool)
        mask_neg[idx_nonzero] = True
        mask_neg[idx_pos] = False

        m_neg = 3
        m_pos = 3
        gamma_sigma = 0.2

        # Randomly generate neg-interaction coefficient according to negative number.
        neg_value = -m_neg * ((1 - gamma_sigma) + 2 * gamma_sigma * np.random.rand(n_neg))
        pos_value = m_pos * ((1 - gamma_sigma) + 2 * gamma_sigma * np.random.rand(n_pos))

        # Scaled individually to the target gamma mean.
        if n_neg > 0:
            neg_value *= gamma_neg_cen[i] / np.mean(np.abs(neg_value))
        if n_pos > 0:
            pos_value *= gamma_pos_cen[i] / np.mean(pos_value)
        if n_nonzero > 0:
            gamma_mean[i] = (np.sum(np.abs(neg_value))+np.sum(pos_value)) / n_nonzero # Only calculate non zero interaction.

        gammas = np.zeros(n_edge)
        gammas[mask_neg] = neg_value
        gammas[mask_pos] = pos_value

        Xnum[i, NumSpecies:NumSpecies + featgam] = gammas

        if n_nonzero > 0:
            den_arr[i] = n_nonzero / n_edge
            pos_arr[i] = n_pos / n_nonzero
        else:
            den_arr[i] = 0
            pos_arr[i] = 0

    Xnum[:, (NumSpecies + featgam):(NumSpecies + featgam + NumSpecies)] = \
        1e-06 / NumSpecies * (0.5 + 0.1 * np.random.randn(datasize, NumSpecies))
    
    # ===== padding =====
    Xnum_pad = prep.pad_Xnum_to_maxspecies(Xnum, NumSpecies, max_species)
    #Xnum_pad = prep.pad_Xnum_to_maxspecies(Xnum, NumSpecies, 20)

    richness_col = np.full(datasize, NumSpecies)

    gamma_neg_arr = gamma_neg_cen
    gamma_pos_arr = gamma_pos_cen
    gamma_mean_arr = gamma_mean

    relative_entropy = epredictor.predict_array_MD(Xnum_pad, device)
    entropy = relative_entropy*(NumSpecies-1)+1
    relative_diversity = dpredictor.predict_array_MD(Xnum_pad, device)
    diversity = relative_diversity*(NumSpecies-1)+1
    num_stable_states = npredictor.predict_array_NS(Xnum_pad, device)
    fluctuation = fpredictor.predict_array_Fl(Xnum_pad, device)
    swi_species = spredictor.predict_array_SS(Xnum_pad, device)

    extra = np.column_stack([richness_col, den_arr, pos_arr, gamma_neg_arr, gamma_pos_arr, gamma_mean_arr,
                                relative_entropy, entropy, relative_diversity, diversity, num_stable_states,
                                fluctuation, swi_species])

    X_full = np.concatenate([Xnum_pad, extra], axis=1)
    all_data.append(X_full)

    print(f"Processed NumSpecies = {NumSpecies}")


Processed NumSpecies = 3
Processed NumSpecies = 4
Processed NumSpecies = 5
Processed NumSpecies = 6
Processed NumSpecies = 7
Processed NumSpecies = 8
Processed NumSpecies = 9
Processed NumSpecies = 10
Processed NumSpecies = 11
Processed NumSpecies = 12
Processed NumSpecies = 13
Processed NumSpecies = 14
Processed NumSpecies = 15
Processed NumSpecies = 16
Processed NumSpecies = 17
Processed NumSpecies = 18
Processed NumSpecies = 19
Processed NumSpecies = 20


In [6]:
# ===== Merge all =====
all_data = np.vstack(all_data)

df = pd.DataFrame(all_data)
n_feature_cols = max_species**2+max_species
df.columns = list(df.columns[:n_feature_cols]) + ["richness", "density", "pos_inall", "gamma_neg_mean", "gamma_pos_mean", 
                                                  "gamma_mean", "relative_H", "H", "relative_D", "D", "states_number", 
                                                  "fluctuation"]+ [f"switching_sp{i+1}" for i in range(max_species)]
df.to_csv(save_path, index=False)

print(f"\ndf size = {df.shape}")
print(f"\nSaved ALL data to: {save_path}")



df size = (144000, 452)

Saved ALL data to: gamma5_fluc2.csv


In [6]:
# Fix mu, D, randomize richness and gamma under different density and positive proportion conditions.
save_path = "gamma5_allneg.csv"

datasize = 4000
max_species = 20
all_data = []

for NumSpecies in range(3, max_species + 1):

    featgam = NumSpecies**2 - NumSpecies
    total_dim = NumSpecies + featgam + NumSpecies
    density = 0.5 + 0.5*np.random.rand(datasize)
    pos_inall = 0
    gamma_neg_cen = 0.01 + 1.49*np.random.rand(datasize)
    gamma_pos_cen = 0.1 + 6.9*np.random.rand(datasize)
    gamma_mean = np.zeros(datasize)
    n_edge = NumSpecies * (NumSpecies - 1) # Upper boundary of the number of interaction.
    Xnum = np.zeros((datasize, total_dim))

    den_arr = np.zeros(datasize)
    pos_arr = np.zeros(datasize)

    Xnum[:, 0:NumSpecies] = 0.5 + 0.1*np.random.randn(datasize, NumSpecies)

    for i in range(datasize):

        n_nonzero = int(round(density[i] * n_edge)) # None zero interaction number.
        n_pos = 0 # Positive interaction number.
        n_neg = n_nonzero - n_pos

        idx_nonzero = np.random.choice(n_edge, n_nonzero, replace=False) # None zero interaction index(from 0).
        idx_pos = np.random.choice(idx_nonzero, n_pos, replace=False)

        mask_pos = np.zeros(n_edge, dtype=bool)
        mask_pos[idx_pos] = True

        mask_neg = np.zeros(n_edge, dtype=bool)
        mask_neg[idx_nonzero] = True
        mask_neg[idx_pos] = False

        m_neg = 3
        m_pos = 3
        gamma_sigma = 0.2

        # Randomly generate neg-interaction coefficient according to negative number.
        neg_value = -m_neg * ((1 - gamma_sigma) + 2 * gamma_sigma * np.random.rand(n_neg))
        pos_value = m_pos * ((1 - gamma_sigma) + 2 * gamma_sigma * np.random.rand(n_pos))

        # Scaled individually to the target gamma mean.
        if n_neg > 0:
            neg_value *= gamma_neg_cen[i] / np.mean(np.abs(neg_value))
        if n_pos > 0:
            pos_value *= gamma_pos_cen[i] / np.mean(pos_value)
        if n_nonzero > 0:
            gamma_mean[i] = (np.sum(np.abs(neg_value))+np.sum(pos_value)) / n_nonzero # Only calculate non zero interaction.

        gammas = np.zeros(n_edge)
        gammas[mask_neg] = neg_value
        gammas[mask_pos] = pos_value

        Xnum[i, NumSpecies:NumSpecies + featgam] = gammas

        if n_nonzero > 0:
            den_arr[i] = n_nonzero / n_edge
            pos_arr[i] = n_pos / n_nonzero
        else:
            den_arr[i] = 0
            pos_arr[i] = 0

    Xnum[:, (NumSpecies + featgam):(NumSpecies + featgam + NumSpecies)] = \
        1e-06 / NumSpecies * (0.5 + 0.1 * np.random.randn(datasize, NumSpecies))
    
    # ===== padding =====
    #Xnum_pad = prep.pad_Xnum_to_maxspecies(Xnum, NumSpecies, max_species)
    Xnum_pad = prep.pad_Xnum_to_maxspecies(Xnum, NumSpecies, 20)

    richness_col = np.full(datasize, NumSpecies)

    gamma_neg_arr = gamma_neg_cen
    gamma_pos_arr = gamma_pos_cen
    gamma_mean_arr = gamma_mean

    relative_entropy = epredictor.predict_array_MD(Xnum_pad, device)
    entropy = relative_entropy*(NumSpecies-1)+1
    relative_diversity = dpredictor.predict_array_MD(Xnum_pad, device)
    diversity = relative_diversity*(NumSpecies-1)+1
    num_stable_states = npredictor.predict_array_NS(Xnum_pad, device)
    fluctuation = fpredictor.predict_array_Fl(Xnum_pad, device)
    swi_species = spredictor.predict_array_SS(Xnum_pad, device)

    extra = np.column_stack([richness_col, den_arr, pos_arr, gamma_neg_arr, gamma_pos_arr, gamma_mean_arr,
                                relative_entropy, entropy, relative_diversity, diversity, num_stable_states,
                                fluctuation, swi_species])

    X_full = np.concatenate([Xnum_pad, extra], axis=1)
    all_data.append(X_full)

    print(f"Processed NumSpecies = {NumSpecies}")

# ===== Merge all =====
all_data = np.vstack(all_data)

df = pd.DataFrame(all_data)
n_feature_cols = max_species**2+max_species
df.columns = list(df.columns[:n_feature_cols]) + ["richness", "density", "pos_inall", "gamma_neg_mean", "gamma_pos_mean", 
                                                  "gamma_mean", "relative_H", "H", "relative_D", "D", "states_number", 
                                                  "fluctuation"]+ [f"switching_sp{i+1}" for i in range(max_species)]
df.to_csv(save_path, index=False)

print(f"\ndf size = {df.shape}")
print(f"\nSaved ALL data to: {save_path}")


Processed NumSpecies = 3
Processed NumSpecies = 4
Processed NumSpecies = 5
Processed NumSpecies = 6
Processed NumSpecies = 7
Processed NumSpecies = 8
Processed NumSpecies = 9
Processed NumSpecies = 10
Processed NumSpecies = 11
Processed NumSpecies = 12
Processed NumSpecies = 13
Processed NumSpecies = 14
Processed NumSpecies = 15
Processed NumSpecies = 16
Processed NumSpecies = 17
Processed NumSpecies = 18
Processed NumSpecies = 19
Processed NumSpecies = 20

df size = (72000, 452)

Saved ALL data to: gamma5_allneg.csv
